In [5]:
import openai
import langchain
import pinecone
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.llms import OpenAI

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
# Lets Read the documents

def read_doc(directory):
    file=PyPDFDirectoryLoader(directory)
    
    docs=file.load()
    
    return docs

In [8]:
docs=read_doc("Docs/")

In [9]:
print(docs[0].page_content)

- 1 - 
 
 
 
 
Budget Speech 2019-20 
 
PART – I 
 
BismillahirRehmanir Raheem 
 
 
Mr. Speaker, 
1. I would like to start by thanking Almighty Allah, the most gracious, 
the most merciful ,as I  present the first Annual Budget of the democratic 
government. A new journey has begun under the leadership of Prime 
Minister Imran Khan. Tehreek-e-Insaaf brings a new vision, a new 
commitment, a new Pakistan.  
2. 22 years of hard work,  and the will of the people of Pakistan have 
brought us here today . It is now time to improve the lives of citizens, to 
remove corruption from public life, to inject merit in our institutions, and to 
lead the economy, to remember the forgotten and to fulfil the destiny of our 
people. 
3. In 1947 our forefather s came together to convert Iqbal’s vision into 
reality and Pakistan emerged as a reality. In 1973, another generation came 
together and gave our nation its Constitution. Now we are the custodians of 
both our country and its rule of law. 
Econom

In [10]:
len(docs)

45

## Divide the docs into chunks

In [11]:
def text_chunks(docs, chunk_size=500, chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    
    chunks=text_splitter.split_documents(docs)
    
    return chunks

In [12]:
chunks=text_chunks(docs)

In [13]:
len(chunks)

171

### Embedding Technique of OpenAI

In [ ]:
embeddings=OpenAIEmbeddings(api_key=os.environ["OPENAI_API_KEY"])


In [24]:
embed=embeddings.embed_query("How are you?")
embed

[-0.01678639088460354,
 -0.012108419068213666,
 0.006708384944812091,
 -0.02596953506456217,
 -0.01614455875761822,
 0.01760102432705513,
 -0.011133327128354463,
 -0.00991754790804512,
 -0.018131772022835158,
 -0.0104236081050627,
 0.02787034767204597,
 0.0016385245805237718,
 -0.007300845799562804,
 -0.011682587654238609,
 0.00723913124360826,
 -0.015391639405459892,
 0.028364064119682324,
 -0.011812188640838309,
 0.01403391731181477,
 -0.020588016099571144,
 0.0024192151568987455,
 0.00635043996148219,
 0.0010391208324074062,
 -0.0081957084907182,
 -0.015910042420536123,
 -0.00778222036046308,
 0.02509318660049475,
 -0.012423164188338286,
 0.02234071302770598,
 -0.025179587258227883,
 0.005628378120131776,
 0.0076896482937006206,
 -0.013194597301923308,
 0.004073168609241797,
 0.00882519873096744,
 -0.02234071302770598,
 0.004057739853837839,
 -0.010479152183310492,
 0.020291784740873214,
 -0.006356611370511515,
 0.027055714229594383,
 0.0012558936118306028,
 -0.005205632643502024,
 

In [25]:
len(embed)

1536

### Vector Search DB in Pinecone

In [ ]:
# Set env variable for LangChain
os.environ["PINECONE_API_KEY"] = ""

In [20]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

In [21]:
index=pc.Index("testing")

In [22]:
index

In [42]:
pip install -qU langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [23]:
docsearch=PineconeVectorStore.from_texts(
    [ch.page_content for ch in chunks],
    embeddings,
    index_name="testing"
)

In [24]:
docsearch

In [25]:
### Cosine Similarity Retrive results

def retrieve_query(query,k=3):
    matching_results=docsearch.similarity_search(query,k=k)
    
    return matching_results

In [38]:
from langchain_community.chat_models import ChatOpenAI

In [ ]:
from langchain.chains.question_answering import load_qa_chain

llm=ChatOpenAI(model_name="gpt-4o-mini-2024-07-18", temperature=.4)

chain=load_qa_chain(llm, chain_type="stuff")

In [47]:
def retrieve_answers(query):
    similar_results=retrieve_query(query)
    print(similar_results,"\n\n")
    response=chain.run(input_documents=similar_results, question=query)
    
    return response

In [48]:
query="what is budget specified for agriculture?"
answer=retrieve_answers(query)

print(answer)

[Document(id='46e9cc31-b6a4-4288-9b02-0d2eeab67b02', metadata={}, page_content='will be implemented \nb. Increase in yields of wheat, rice, sugarcane, cotton  – \nRs.44.8 billion shall be provided for this purpose \nc. Harnessing untapped potential in fisheries through \nshrimp farming, cold water trout farming etc.  – \nRs.9.3 billion of shall be spent on these projects \nd. Undertaking livestock initiatives for small and \nmedium farmers – Rs.5.6 billion will be provided for \nbackyard poultry and save the buffalo calf programme'), Document(id='a8934135-1ab4-44c8-8d06-7726f3a5cd96', metadata={}, page_content='e. In addition, in the budget 2019-20, the proposals are for: \n• Continuation of subsidy to agriculture tube \nwells - Agriculture sector tube -wells shall be \ncharged ata subsidized rate of 6.85. In \nBalochistan, a flat rate of Rs.10,000 per month is \ncharged from the farmers and excess bill up to \nRs.75,000 per month is shared by the Federal and \nprovincial Governments \